# 4. Conformers and potential-energy surfaces
**Kernel:** AIMNet2. Complete the README preflight first; run cells in order from a fresh kernel.

**Learning goals:** relate conformers to torsional minima; distinguish rigid and relaxed scans;
use independent scan rows as a parallel workload. Start with the coarse live scan. Use supplied
reference results for the dense 2D relaxed surface, or submit your own batch job.

![Alanine dipeptide](alanine_dipeptide.png)

**Predict:** can two different starting conformers relax to the same minimum? Will a rigid torsional
scan generally have higher energies than a relaxed scan at the same angles?


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in the repository, workshop folder, or exercise folder.
_candidates = [Path.cwd(), *Path.cwd().parents]
WORKSHOP = next((p for base in _candidates for p in (base, base / 'workshop_demo')
                 if (p / 'workshop_utils.py').is_file()), None)
if WORKSHOP is None:
    raise RuntimeError('Launch Jupyter from the repository or workshop_demo folder.')
if str(WORKSHOP) not in sys.path:
    sys.path.insert(0, str(WORKSHOP))
from workshop_utils import start_exercise, mace_model, relax, smoke_check, signed_angle
DATA, OUTPUT = start_exercise('ConformerSampling')
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.visualize import view


## 1. Load structures and initialize the calculator
The supplied XYZ contains multiple conformers of the same molecule. Each copy needs a calculator
before energy or force evaluation. Charge is zero for this neutral molecule.
Model access should be checked before the workshop; first-time loading may need a download.


In [ ]:
import os
os.environ['TORCH_COMPILE_DISABLE'] = '1'
from aimnet.calculators import AIMNet2ASE
from ase.optimize import BFGS
from ase.constraints import FixInternals
calculator = AIMNet2ASE('aimnet2', charge=0)
mols = read(DATA / 'alanine_dipeptide_conformers.xyz', index=':')
mols[0].calc = calculator
smoke_check(mols[0])
view(mols[0], viewer='x3d')


## 2. Identify the two torsions
Atom indices are zero-based. A torsion is defined by four atoms; the rotation list identifies which
part of the molecule moves. These lists are specific to the supplied atom ordering, not arbitrary XYZ files.
ASE reports dihedrals in [0, 360); plots below wrap them into [-180, 180).
Print the atom indices and elements to connect these lists to the structure.


In [ ]:
phi_indices = [6, 4, 3, 1]
phi_rotation_indices = [13, 0, 1, 2, 10, 11, 12]
psi_indices = [3, 4, 6, 8]
psi_rotation_indices = [7, 8, 9, 18, 19, 20, 21]
print(list(enumerate(mols[0].get_chemical_symbols())))
print('Initial phi, psi:', signed_angle(mols[0].get_dihedral(*phi_indices)),
      signed_angle(mols[0].get_dihedral(*psi_indices)))


## 3. Relax a few conformers
Start with three to keep the live work bounded. The reported energy is a potential energy, not a
free energy or a population. Check convergence before comparing the energies.


In [ ]:
opt_mols = []
for i, original in enumerate(mols[:3]):
    mol = original.copy()
    mol.calc = calculator
    ok = relax(mol, BFGS, fmax=0.05, steps=150, logfile=str(OUTPUT / f'conformer_{i}.log'))
    mol.info['converged'] = ok
    opt_mols.append(mol)
    print(i, mol.get_potential_energy(), 'eV')
write(OUTPUT / 'relaxed_conformers.extxyz', opt_mols)
view(opt_mols[0], viewer='x3d')


## 4. Make a coarse rigid scan
Only the torsions change; other internal coordinates are not optimized. Each grid point starts from
the same initial geometry. The default 30-degree grid has 144 points; 10 degrees has 1296 points.
We omit the duplicate +180 endpoint because torsion angles are periodic.
High-energy regions can contain close contacts. Treat them as candidates for inspection, not reliable chemistry.


In [ ]:
STEP = 30
angles = np.arange(-180, 180, STEP)
rigid_data = []
for phi in angles:
    for psi in angles:
        mol = mols[0].copy()
        mol.set_dihedral(*phi_indices, float(phi), indices=phi_rotation_indices)
        mol.set_dihedral(*psi_indices, float(psi), indices=psi_rotation_indices)
        mol.calc = calculator
        rigid_data.append((phi, psi, mol.get_potential_energy()))
    print(f'Completed phi={phi} degrees')
rigid_data = np.array(rigid_data)
np.savetxt(OUTPUT / 'rigid_scan.csv', rigid_data, delimiter=',',
           header='phi_deg,psi_deg,energy_eV', comments='')
plt.figure(figsize=(7, 5))
plt.tricontourf(rigid_data[:, 0], rigid_data[:, 1], rigid_data[:, 2]-rigid_data[:, 2].min(), levels=30)
plt.xlabel('Phi (degrees)'); plt.ylabel('Psi (degrees)')
plt.colorbar(label='Energy relative to sampled minimum (eV)')
plt.title('Rigid scan: coarse grid'); plt.show()


## 5. Prepare a relaxed row for a batch scan
Hold psi at -180 degrees and step through phi. Reusing the preceding relaxed structure can help
convergence but can also follow one local branch of the surface. It does not guarantee a global minimum.

This section creates the input for the parallel extension. If time is short, skip to section 7 and
use the supplied reference surface. A failed relaxation stops row preparation rather than silently
passing an unconverged input to the batch job.


In [ ]:
phi_structures = []
mol = mols[0].copy()
mol.calc = calculator
mol.set_dihedral(*psi_indices, -180.0, indices=psi_rotation_indices)
for phi in angles:
    mol.set_constraint()
    mol.set_dihedral(*phi_indices, float(phi), indices=phi_rotation_indices)
    mol.set_constraint(FixInternals(dihedrals_deg=[(float(phi), phi_indices), (-180.0, psi_indices)]))
    ok = relax(mol, BFGS, fmax=0.05, steps=150,
               logfile=str(OUTPUT / f'phi_{int(phi)}.log'))
    if not ok:
        raise RuntimeError(f'Row preparation failed at phi={phi}; inspect the log and geometry.')
    mol.info.update(target_phi=float(phi), target_psi=-180.0,
                    actual_phi=mol.get_dihedral(*phi_indices), actual_psi=mol.get_dihedral(*psi_indices),
                    Energy=mol.get_potential_energy(), converged=True)
    phi_structures.append(mol.copy())
write(OUTPUT / 'phi_1D_scan_structures.xyz', phi_structures, format='extxyz')
print('Batch input:', OUTPUT / 'phi_1D_scan_structures.xyz')


## 6. Optional HPRC extension: parallelize independent rows
Each phi row scans psi sequentially. Rows can run independently, but points within a row depend
on the previous relaxation. This is why distributing rows is a useful parallelization strategy.

From a terminal in this folder, activate AIMNet2 in your allocated compute environment and run:

```bash
python parallel_relaxed_scan.py --input results/YOUR_RUN/phi_1D_scan_structures.xyz --output results/YOUR_RUN/Relaxed_2D_scan.xyz --workers 4 --step 30
```

Replace `YOUR_RUN` with the directory printed above. Four workers require four allocated CPUs.
For Slurm, adapt `submit_scan.slurm` to the site's environment activation, then use
`sbatch submit_scan.slurm INPUT_PATH OUTPUT_PATH`. The script also writes completed rows alongside
the output so a failed job does not discard every completed row. See the README for recovery.

**Discuss:** why might four workers take more than one quarter of the serial time? Each process
loads its own model, so memory and startup costs matter too.


## 7. Inspect a relaxed surface
The default below reads supplied results from the original demo. Their model/software provenance
and convergence have not been revalidated; they are labeled reference data, not your fresh calculation.
To inspect your own batch job, set `SURFACE` to its completed output and change `SURFACE_LABEL`.
The loader excludes explicitly failed points and wraps periodic endpoint duplicates.


In [ ]:
SURFACE = DATA / 'reference_results' / 'Relaxed_2D_scan.xyz'
SURFACE_LABEL = 'Supplied reference (provenance not revalidated)'
structures = read(SURFACE, index=':')
points = {}
for mol in structures:
    if not mol.info.get('converged', True):
        continue
    key = (round(signed_angle(float(mol.info['target_phi'])), 6),
           round(signed_angle(float(mol.info['target_psi'])), 6))
    energy = float(mol.info['Energy'])
    points[key] = min(points.get(key, float('inf')), energy)
if len(points) < 3:
    raise ValueError('Too few valid surface points to plot.')
surface = np.array([(phi, psi, energy) for (phi, psi), energy in points.items()])
plt.figure(figsize=(7, 5))
plt.tricontourf(surface[:, 0], surface[:, 1], surface[:, 2]-surface[:, 2].min(), levels=30)
plt.colorbar(label='Energy relative to sampled minimum (eV)')
for mol in globals().get('opt_mols', []):
    if mol.info['converged']:
        plt.scatter(signed_angle(mol.get_dihedral(*phi_indices)),
                    signed_angle(mol.get_dihedral(*psi_indices)), c='white', edgecolors='black')
plt.xlabel('Phi (degrees)'); plt.ylabel('Psi (degrees)')
plt.title(SURFACE_LABEL); plt.show()


## Try, explain, and report
- Do relaxed conformers lie near low-energy regions of the relaxed surface?
- Compare rigid and relaxed scans. Their separately shifted color scales do not show absolute energy differences.
- Reverse the scan direction as an extension. Does continuation follow a different local minimum?
- Why is a potential-energy surface insufficient by itself to predict room-temperature populations?

**Checkpoint:** report grid spacing, convergence threshold, whether data were supplied or calculated,
and one physical conclusion plus one limitation. Save the model identifier with any results you share.
